# Week 11 live coding: MT evaluation

Mục tiêu: đọc một dataset nhỏ gồm Chinese source, Vietnamese MT output, Vietnamese reference, rồi so sánh automatic metrics với human review labels.

Core tuần này không yêu cầu bạn tự thiết kế metric. Bạn chỉ cần chạy notebook, đọc bảng, và viết kết quả cẩn thận.

In [1]:
import hashlib
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

try:
    import pandas as pd
    import matplotlib.pyplot as plt
    import sacrebleu
    from sacrebleu.metrics import BLEU, CHRF, TER
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas==2.3.3", "matplotlib==3.9.4", "sacrebleu==2.5.1"])
    import pandas as pd
    import matplotlib.pyplot as plt
    import sacrebleu
    from sacrebleu.metrics import BLEU, CHRF, TER

CANDIDATES = [Path("."), Path("weeks/week-11-mt-evaluation")]
WEEK_DIR = next(
    candidate for candidate in CANDIDATES
    if (candidate / "data" / "raw" / "week11_mt_evaluation_segments.csv").exists()
)
DATA_PATH = WEEK_DIR / "data" / "raw" / "week11_mt_evaluation_segments.csv"
TABLE_DIR = WEEK_DIR / "outputs" / "tables"
FIG_DIR = WEEK_DIR / "outputs" / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SHA = "2b87763883f56fe746262cb7be8b1b363922cde8b2ec31b1121139e97388268c"
REMOTE_DATA = "https://raw.githubusercontent.com/mtuann/tcsol-python-research/main/weeks/week-11-mt-evaluation/data/raw/week11_mt_evaluation_segments.csv"
if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(REMOTE_DATA, DATA_PATH)

actual_sha = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print("Data file:", DATA_PATH)
print("SHA-256:", actual_sha)
if actual_sha != EXPECTED_SHA:
    print("Note: SHA differs from the course snapshot. Continue only if you intentionally changed the data.")

Data file: weeks/week-11-mt-evaluation/data/raw/week11_mt_evaluation_segments.csv
SHA-256: 2b87763883f56fe746262cb7be8b1b363922cde8b2ec31b1121139e97388268c


## 1. Read source, MT output, and reference

Một source segment xuất hiện 3 lần vì có 3 synthetic MT profiles. Reference là bản tiếng Việt dùng để so metric, không phải “đáp án duy nhất” cho mọi cách dịch hợp lệ.

In [2]:
data = pd.read_csv(DATA_PATH)
print("Rows:", len(data))
print("Source segments:", data["segment_id"].nunique())
print("Systems:", ", ".join(sorted(data["mt_system"].unique())))

preview_cols = ["segment_id", "mt_system", "zh_source", "vi_mt_output", "vi_reference", "error_type", "severity"]
print(data.loc[data["segment_id"].eq("S001"), preview_cols].to_string(index=False))

Rows: 36
Source segments: 12
Systems: MT_A, MT_B, MT_C
segment_id mt_system            zh_source                                                                                  vi_mt_output                                                                   vi_reference  error_type severity
      S001      MT_A 推进教育数字化有助于扩大优质资源覆盖面。                   Thúc đẩy giáo dục số hóa có ích cho mở rộng mặt bao phủ tài nguyên ưu chất. Thúc đẩy số hóa giáo dục giúp mở rộng độ bao phủ của nguồn lực chất lượng cao. terminology    major
      S001      MT_B 推进教育数字化有助于扩大优质资源覆盖面。 Thúc đẩy chuyển đổi số trong giáo dục giúp mở rộng phạm vi tiếp cận nguồn lực chất lượng cao. Thúc đẩy số hóa giáo dục giúp mở rộng độ bao phủ của nguồn lực chất lượng cao.       style    minor
      S001      MT_C 推进教育数字化有助于扩大优质资源覆盖面。                                        Số hóa giáo dục giúp mở rộng nguồn lực chất lượng cao. Thúc đẩy số hóa giáo dục giúp mở rộng độ bao phủ của nguồn lực chất lượng cao.    omission    major


## 2. Compute corpus-level metrics by system

BLEU và chrF++: cao hơn thường tốt hơn. TER: thấp hơn thường tốt hơn. Trong tuần này, TER là số edits cần để match một reference, không phải thời gian MTPE thật. Week 12 mới đo post-editing effort bằng time log và post-edited text.

In [3]:
bleu = BLEU(effective_order=True)
chrf = CHRF(word_order=2)
ter = TER()

metric_rows = []
for system, rows in data.groupby("mt_system"):
    hypotheses = rows["vi_mt_output"].tolist()
    references = [rows["vi_reference"].tolist()]
    metric_rows.append({
        "mt_system": system,
        "segments": len(rows),
        "bleu": round(bleu.corpus_score(hypotheses, references).score, 2),
        "chrf_pp": round(chrf.corpus_score(hypotheses, references).score, 2),
        "ter": round(ter.corpus_score(hypotheses, references).score, 2),
        "mean_adequacy": round(rows["human_adequacy_1_5"].mean(), 2),
        "mean_fluency": round(rows["human_fluency_1_5"].mean(), 2),
        "revision_needed_rows": int((rows["severity"] != "none").sum()),
    })

metric_summary = pd.DataFrame(metric_rows).sort_values(["chrf_pp", "bleu"], ascending=False)
metric_summary.to_csv(TABLE_DIR / "week11_system_metric_summary.csv", index=False)
print(metric_summary.to_string(index=False))

metric_settings = pd.DataFrame([
    {"item": "sacrebleu_version", "value": sacrebleu.__version__},
    {"item": "bleu_signature", "value": str(bleu.get_signature())},
    {"item": "chrf_pp_signature", "value": str(chrf.get_signature())},
    {"item": "ter_signature", "value": str(ter.get_signature())},
    {"item": "reference_count", "value": "1 Vietnamese reference per source segment"},
    {"item": "dataset_type", "value": "synthetic classroom dataset; not a real benchmark"},
])
metric_settings.to_csv(TABLE_DIR / "week11_metric_settings.csv", index=False)
print()
print("Metric settings:")
print(metric_settings.to_string(index=False))

mt_system  segments  bleu  chrf_pp   ter  mean_adequacy  mean_fluency  revision_needed_rows
     MT_B        12 68.02    80.73 20.09           4.59          4.43                     6
     MT_C        12 38.12    57.48 43.93           3.21          3.54                    12
     MT_A        12 33.59    55.77 47.20           2.96          2.73                    12

Metric settings:
             item                                                               value
sacrebleu_version                                                               2.5.1
   bleu_signature         nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.5.1
chrf_pp_signature         nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|version:2.5.1
    ter_signature nrefs:1|case:lc|tok:tercom|norm:no|punct:yes|asian:no|version:2.5.1
  reference_count                           1 Vietnamese reference per source segment
     dataset_type                   synthetic classroom dataset; not a real benchmark


## 3. Summarize human review labels

`acceptable_variant` và `no_error` không phải lỗi. Vì vậy ta tách `error_type`, `severity`, và `review_decision` để tránh đọc nhầm.

In [4]:
human_review = (
    data.assign(revision_needed=data["severity"].ne("none"))
    .groupby("mt_system", as_index=False)
    .agg(
        segments=("segment_id", "count"),
        mean_adequacy=("human_adequacy_1_5", "mean"),
        mean_fluency=("human_fluency_1_5", "mean"),
        revision_needed_rows=("revision_needed", "sum"),
        acceptable_or_no_error_rows=("review_decision", lambda s: int(s.isin(["acceptable_variant", "accept_with_minor_edit"]).sum())),
    )
)
human_review[["mean_adequacy", "mean_fluency"]] = human_review[["mean_adequacy", "mean_fluency"]].round(2)
human_review.to_csv(TABLE_DIR / "week11_human_review_summary.csv", index=False)
print(human_review.to_string(index=False))

severity_summary = (
    data.groupby(["mt_system", "severity"], as_index=False)
    .size()
    .rename(columns={"size": "segment_count"})
    .sort_values(["mt_system", "severity"])
)
severity_summary.to_csv(TABLE_DIR / "week11_severity_summary.csv", index=False)
print()
print("Severity summary:")
print(severity_summary.to_string(index=False))

mt_system  segments  mean_adequacy  mean_fluency  revision_needed_rows  acceptable_or_no_error_rows
     MT_A        12           2.96          2.73                    12                            2
     MT_B        12           4.59          4.43                     6                           12
     MT_C        12           3.21          3.54                    12                            2

Severity summary:
mt_system severity  segment_count
     MT_A    major              6
     MT_A    minor              6
     MT_B    minor              6
     MT_B     none              6
     MT_C    major              7
     MT_C    minor              5


## 4. Segment-level helper table

Segment-level scores are noisy, but they are useful as a reading aid. Use them to choose examples for discussion, not to rank systems alone.

In [5]:
segment_rows = []
for _, row in data.iterrows():
    segment_rows.append({
        **row.to_dict(),
        "segment_chrf_pp": round(chrf.sentence_score(row["vi_mt_output"], [row["vi_reference"]]).score, 2),
        "segment_ter": round(ter.sentence_score(row["vi_mt_output"], [row["vi_reference"]]).score, 2),
    })
segment_scores = pd.DataFrame(segment_rows)
sample = segment_scores[segment_scores["segment_id"].isin(["S001", "S004", "S011", "S012"])]
sample.to_csv(TABLE_DIR / "week11_segment_review_sample.csv", index=False)
print(sample[["segment_id", "mt_system", "segment_chrf_pp", "segment_ter", "error_type", "severity", "review_decision"]].to_string(index=False))

segment_id mt_system  segment_chrf_pp  segment_ter  error_type severity        review_decision
      S001      MT_A            40.04        61.11 terminology    major                 revise
      S001      MT_B            63.70        38.89       style    minor accept_with_minor_edit
      S001      MT_C            66.32        33.33    omission    major                 revise
      S004      MT_A            68.25        17.65 terminology    minor                 revise
      S004      MT_B            95.25         5.88    no_error     none     acceptable_variant
      S004      MT_C            77.15        23.53    omission    minor                 revise
      S011      MT_A            73.40        40.00 terminology    minor accept_with_minor_edit
      S011      MT_B            92.80        13.33       style    minor accept_with_minor_edit
      S011      MT_C            73.31        33.33    omission    minor accept_with_minor_edit
      S012      MT_A            62.79        50.00

## 5. Export figures

Figure 1 is Core. Figure 2 is Stretch, useful if you want to explain why automatic metrics and human review should be read together.

In [6]:
plt.rcParams.update({"font.size": 11, "axes.titlesize": 14, "axes.labelsize": 11})

plot = metric_summary.sort_values("chrf_pp")
fig, ax = plt.subplots(figsize=(7.8, 4.8))
ax.barh(plot["mt_system"], plot["chrf_pp"], color="#2f63ea")
ax.set_title("Week 11 MT evaluation: chrF++ by system")
ax.set_xlabel("chrF++ (higher is better)")
ax.set_ylabel("Synthetic MT profile")
for i, value in enumerate(plot["chrf_pp"]):
    ax.text(value + 0.4, i, f"{value:.1f}", va="center", fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "week11_chrf_by_system.png", dpi=200)
fig.savefig(FIG_DIR / "week11_chrf_by_system.svg")
plt.close(fig)

error_counts = data[data["severity"] != "none"].groupby("mt_system").size().reindex(metric_summary["mt_system"]).fillna(0)
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.bar(error_counts.index, error_counts.values, color="#b45309")
ax.set_title("Week 11 human review: rows needing revision")
ax.set_xlabel("Synthetic MT profile")
ax.set_ylabel("Rows with minor/major revision")
for i, value in enumerate(error_counts.values):
    ax.text(i, value + 0.2, str(int(value)), ha="center", fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "week11_error_count_by_system.png", dpi=200)
fig.savefig(FIG_DIR / "week11_error_count_by_system.svg")
plt.close(fig)

domain_summary = (
    segment_scores.assign(revision_needed=segment_scores["severity"].ne("none"))
    .groupby("domain", as_index=False)
    .agg(
        rows=("segment_id", "count"),
        source_segments=("segment_id", "nunique"),
        mean_chrf_pp=("segment_chrf_pp", "mean"),
        mean_ter=("segment_ter", "mean"),
        revision_needed_rows=("revision_needed", "sum"),
    )
)
domain_summary[["mean_chrf_pp", "mean_ter"]] = domain_summary[["mean_chrf_pp", "mean_ter"]].round(2)
domain_summary.to_csv(TABLE_DIR / "week11_domain_summary.csv", index=False)

print("Figures exported:")
print(FIG_DIR / "week11_chrf_by_system.png")
print(FIG_DIR / "week11_error_count_by_system.png")
print()
print("Domain summary exported:")
print(domain_summary.to_string(index=False))

Figures exported:
weeks/week-11-mt-evaluation/outputs/figures/week11_chrf_by_system.png
weeks/week-11-mt-evaluation/outputs/figures/week11_error_count_by_system.png

Domain summary exported:
               domain  rows  source_segments  mean_chrf_pp  mean_ter  revision_needed_rows
           assessment     6                2         65.52     39.44                     6
           curriculum     6                2         72.12     24.51                     4
       digitalization     6                2         58.20     44.44                     6
               equity     6                2         70.25     29.97                     4
international_chinese     3                1         62.47     38.10                     3
policy_implementation     3                1         63.67     43.33                     2
  teacher_development     6                2         60.39     41.01                     5


## 6. Paper-facing writing

Dùng bảng và figure để viết Results paragraph. Paragraph phải có: metric result, human review result, và limitation.

In [7]:
best_chrf = metric_summary.iloc[0]
best_ter = metric_summary.sort_values("ter").iloc[0]
most_revision = human_review.sort_values("revision_needed_rows", ascending=False).iloc[0]

paragraph = (
    f"In the synthetic Week 11 MT evaluation dataset, {best_chrf['mt_system']} produced the highest chrF++ score "
    f"({best_chrf['chrf_pp']}) across {int(best_chrf['segments'])} source segments, and {best_ter['mt_system']} also had the lowest TER "
    f"({best_ter['ter']}). This indicates that its Vietnamese outputs had the strongest surface overlap with the single reference translation. "
    f"The human review table points in the same broad direction: {most_revision['mt_system']} had the largest number of rows needing revision "
    f"({int(most_revision['revision_needed_rows'])}), while the strongest metric profile had fewer major problems and more acceptable variants. "
    "However, the result should be reported as classroom evidence rather than a benchmark claim. The dataset is synthetic, contains only education-policy sentences, and uses one Vietnamese reference per Chinese source segment. "
    "Therefore, BLEU, chrF++, and TER should support, not replace, bilingual review of omission, terminology, style, and word-order issues."
)
print(paragraph)
print()
print("Word count:", len(paragraph.split()))

caption = (
    "Figure 1 compares chrF++ across three synthetic MT profiles for 12 Chinese-Vietnamese education-policy source segments "
    "(36 system-segment rows). Higher chrF++ indicates stronger overlap with the single Vietnamese reference, but the score does not replace bilingual human review."
)
source_note = (
    "Metric source note: BLEU/chrF++/TER were computed with sacrebleu, version " + sacrebleu.__version__ +
    ". Report metric signatures from week11_metric_settings.csv and state that the dataset is synthetic. Access date: 2026-06-04."
)
print()
print("Caption:")
print(caption)
print()
print("Source note:")
print(source_note)

In the synthetic Week 11 MT evaluation dataset, MT_B produced the highest chrF++ score (80.73) across 12 source segments, and MT_B also had the lowest TER (20.09). This indicates that its Vietnamese outputs had the strongest surface overlap with the single reference translation. The human review table points in the same broad direction: MT_A had the largest number of rows needing revision (12), while the strongest metric profile had fewer major problems and more acceptable variants. However, the result should be reported as classroom evidence rather than a benchmark claim. The dataset is synthetic, contains only education-policy sentences, and uses one Vietnamese reference per Chinese source segment. Therefore, BLEU, chrF++, and TER should support, not replace, bilingual review of omission, terminology, style, and word-order issues.

Word count: 125

Caption:
Figure 1 compares chrF++ across three synthetic MT profiles for 12 Chinese-Vietnamese education-policy source segments (36 syste